In [1]:
import numpy as np
from tqdm import tqdm

# ---------------------------------------------------------
# Spike size vs. bulk edge and signal (setting of Fig. 1c)
# ---------------------------------------------------------
# Setting of Fig. 1 (alpha=1.0, eta_a=eta_f=0.8, lambda=0.01, teacher tanh,
# student ReLU), with 5 seeds per configuration. Configurations: base and
# half the dimension (n=500). At checkpoints tau we record the
# spectrum of F_perp F_perp^T (= n_perp G_t), the top eigenvector's
# participation ratio (delocalization check), the bulk level (mean r_i^2),
# and the signal level (mean m_i^2).

lambda_ = 0.01


def teacher_func(z):
    return np.tanh(z)


def sigma(x):
    return np.maximum(x, 0)


def sigma_prime(x):
    return (x > 0).astype(float)


def mp_edge(k, n):
    # Upper bulk edge of F_perp F_perp^T at initialization: rows of F are
    # N(0, I/n) projected onto the (n-1)-dim complement, so the nonzero
    # spectrum follows Marchenko-Pastur with ratio k/(n-1), scaled by (n-1)/n.
    n_perp = n - 1
    gamma = k / n_perp
    return (n_perp / n) * (1 + np.sqrt(gamma)) ** 2


def run_sim(n, alpha=1.0, eta=0.8, tau=2.0, seed=42, checkpoints=(0.5, 1.0, 2.0)):
    k = int(alpha * n)
    steps = int(tau * n)
    lr_a = eta / k
    lr_f = eta / k

    rng = np.random.default_rng(seed)
    xi = rng.normal(0, 1, n)
    xi /= np.linalg.norm(xi)
    F = rng.normal(0, 1 / np.sqrt(n), (k, n))
    a = rng.normal(0, 1, k)

    cp_steps = {int(round(c * n)): c for c in checkpoints}
    records = {}

    for t in tqdm(range(steps), desc=f"n={n}, alpha={alpha}, eta={eta}, seed={seed}"):
        # Forward
        z0 = rng.normal()
        g_train = rng.normal(0, 1, n)
        g_train -= np.dot(g_train, xi) * xi
        x_train = z0 * xi + g_train
        h = F @ x_train
        e = (np.dot(a, sigma(h)) / k) - teacher_func(z0)
        v_t = e * a * sigma_prime(h)

        # Update (same convention as the Fig. 1 notebooks)
        a -= lr_a * (e * sigma(h) + lambda_ * a)
        F -= lr_f * np.outer(v_t, x_train)

        if (t + 1) in cp_steps:
            F_perp = F - np.outer(F @ xi, xi)
            Sigma = F_perp @ F_perp.T  # = n_perp * G_t
            w, V = np.linalg.eigh(Sigma)
            eigs = w[::-1]
            top_vec = V[:, -1]
            m = F @ xi
            records[cp_steps[t + 1]] = {
                "eigs": eigs,
                "edge": mp_edge(k, n),
                "pr_frac": 1.0 / np.sum(top_vec**4) / k,  # 1 = fully delocalized
                "bulk_level": np.mean(np.sum(F_perp**2, axis=1)),  # mean r_i^2
                "signal_level": np.mean(m**2),  # mean m_i^2
                "k": k,
            }
    return records

In [2]:
seeds = (42, 1, 2, 3, 4)

configs = {
    "base (n=1000, alpha=1.0, eta=0.8)":
        dict(n=1000, alpha=1.0, eta=0.8, tau=2.0, checkpoints=(0.5, 1.0, 2.0)),
    "half the dimension (n=500)":
        dict(n=500, alpha=1.0, eta=0.8, tau=2.0, checkpoints=(0.5, 1.0, 2.0)),
}

results = {name: [run_sim(seed=s, **kw) for s in seeds] for name, kw in configs.items()}

n=500, alpha=1.0, eta=0.8, seed=4: 100%|██████████| 1000/1000 [00:00<00:00, 1480.25it/s]


In [3]:
# Base-run table, mean +/- std over the 5 seeds. Per-neuron spike variance is
# lam1 / k; we compare it with the bulk level (mean r_i^2, i.e.
# hat c_t) and the signal level (mean m_i^2). pr_frac is the participation
# ratio of the top eigenvector divided by k (1 = fully delocalized).

import numpy as np


def ms(vals, digits=3):
    vals = np.asarray(vals, dtype=float)
    return f"{vals.mean():.{digits}g} ± {vals.std():.2g}"


name = "base (n=1000, alpha=1.0, eta=0.8)"
print(f"### {name}, mean ± std over seeds {seeds}")
print("| tau | lam1 | MP edge (init.) | bulk level | signal level "
      "| (lam1/k)/bulk | (lam1/k)/signal | PR frac |")
print("|" + "---|" * 8)
for c in (0.5, 1.0, 2.0):
    recs = [r[c] for r in results[name]]
    k = recs[0]["k"]
    lam1 = [r["eigs"][0] for r in recs]
    bulk = [r["bulk_level"] for r in recs]
    sig = [r["signal_level"] for r in recs]
    row = [
        f"{c}", ms(lam1), f"{recs[0]['edge']:.3g}", ms(bulk), ms(sig),
        ms([l / k / b for l, b in zip(lam1, bulk)]),
        ms([l / k / s for l, s in zip(lam1, sig)]),
        ms([r["pr_frac"] for r in recs], 2),
    ]
    print("| " + " | ".join(row) + " |")

### base (n=1000, alpha=1.0, eta=0.8), mean ± std over seeds (42, 1, 2, 3, 4)
| tau | lam1 | MP edge (init.) | bulk level | signal level | (lam1/k)/bulk | (lam1/k)/signal | PR frac |
|---|---|---|---|---|---|---|---|
| 0.5 | 29.1 ± 1.7 | 4 | 1.05 ± 0.0033 | 0.0136 ± 0.00092 | 0.0276 ± 0.0016 | 2.14 ± 0.073 | 0.33 ± 0.016 |
| 1.0 | 45.7 ± 2.1 | 4 | 1.1 ± 0.0049 | 0.049 ± 0.0042 | 0.0417 ± 0.0018 | 0.936 ± 0.048 | 0.32 ± 0.017 |
| 2.0 | 63.3 ± 1.6 | 4 | 1.15 ± 0.0033 | 0.161 ± 0.01 | 0.0549 ± 0.0015 | 0.395 ± 0.032 | 0.33 ± 0.016 |


In [4]:
# Dimension check, mean ± std over seeds (per-seed ratios, then aggregated):
# lam1(n=500)/lam1(n=1000) at equal tau; 0.5 means lam1/k is unchanged.

base = results["base (n=1000, alpha=1.0, eta=0.8)"]
half_n = results["half the dimension (n=500)"]

print("dimension: lam1(n=500)/lam1(n=1000) at equal tau")
for c in (0.5, 1.0, 2.0):
    ratios = [hn[c]["eigs"][0] / b[c]["eigs"][0] for hn, b in zip(half_n, base)]
    print(f"  tau={c}: {ms(ratios)}")

dimension: lam1(n=500)/lam1(n=1000) at equal tau
  tau=0.5: 0.543 ± 0.037
  tau=1.0: 0.53 ± 0.026
  tau=2.0: 0.518 ± 0.022
